# Atelier Preparation de Donnees Textuelles

## Partie 1 – Exploration du corpus

### 1) Charger les données CSV

In [1]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt

pd.set_option('display.max_colwidth', 120)

df = pd.read_csv('../data/smart_reviews_raw.csv')
df.head()

,id_avis,date,source,produit,texte,sentiment,note,langue
0,AV0001,2026-02-27,mobile,Ordinateur NovaBook,"Très bonne expérience, simple et efficace.",positif,4,fr
1,AV0002,2026-01-09,web,SmartPhone X,Très satisfait de mon achat 👍 #avis,positif,4,fr
2,AV0003,2026-07-03,réseaux_sociaux,Écouteurs AirSound,"Produit parfait, rien à signaler.",positif,4,fr
3,AV0004,2026-06-28,sav,SmartWatch Pro,"Produit excellent, je suis très satisfait. !!!",positif,5,fr
4,AV0005,2026-01-24,sav,SmartPhone X,LA BATTERIE TIENT VRAIMENT BIEN ET L'ÉCRAN EST SUPERBE.,positif,5,fr


### 2) Combien d'avis contient le dataset ?

In [2]:
nb_avis = df.shape[0]
print(f"Le dataset contient {nb_avis} avis.")

Le dataset contient 1200 avis.


### 3) Combien de colonnes possède-t-il ?

In [3]:
nb_colonnes = df.shape[1]
print(f"Le dataset possède {nb_colonnes} colonnes : {list(df.columns)}")

Le dataset possède 8 colonnes : ['id_avis', 'date', 'source', 'produit', 'texte', 'sentiment', 'note', 'langue']


### 4) Quel est le type de chaque colonne ?

In [4]:
df.dtypes

id_avis        str
date           str
source         str
produit        str
texte          str
sentiment      str
note         int64
langue         str
dtype: object

### 5) Existe-t-il des valeurs manquantes ?

In [5]:
df.isna().sum()

id_avis      0
date         0
source       0
produit      0
texte        5
sentiment    0
note         0
langue       0
dtype: int64

### 6) Identifier quelques types de texte

In [6]:
url_re = re.compile(r'https?://\S+|www\.\S+')
mention_re = re.compile(r'@\w+')
hashtag_re = re.compile(r'#\w+')
emoji_re = re.compile(r'[\U0001F300-\U0001FAFF\U00002600-\U000027BF]')
ponct_re = re.compile(r'[!?.,;:]{3,}')
repetition_re = re.compile(r'(.)\1{2,}')

texte_series = df['texte'].fillna('')

exemples = {
    'texte normal': texte_series[texte_series.str.match(r'^[A-Za-zÀ-ÿ ,.\'\-]+$', na=False)].iloc[0] if not texte_series[texte_series.str.match(r'^[A-Za-zÀ-ÿ ,.\'\-]+$', na=False)].empty else None,
    'texte vide': df.loc[df['texte'].isna() | (df['texte'].str.strip() == ''), 'texte'].iloc[0] if df['texte'].isna().any() else '(aucun trouve)',
    'texte avec URL': texte_series[texte_series.str.contains(url_re)].iloc[0],
    'texte avec mention': texte_series[texte_series.str.contains(mention_re)].iloc[0],
    'texte avec hashtag': texte_series[texte_series.str.contains(hashtag_re)].iloc[0],
    'texte avec emojis': texte_series[texte_series.str.contains(emoji_re)].iloc[0],
    'texte avec beaucoup de ponctuation': texte_series[texte_series.str.contains(ponct_re)].iloc[0],
    'texte en majuscules': texte_series[texte_series.str.isupper() & (texte_series.str.len() > 5)].iloc[0],
    'texte avec repetition de caracteres': texte_series[texte_series.str.contains(repetition_re)].iloc[0],
}

for cle, valeur in exemples.items():
    print(f"- {cle} : {valeur!r}")

- texte normal : 'Très  bonne  expérience,  simple  et  efficace.'
- texte vide : nan
- texte avec URL : 'Produit parfait, rien à signaler. https://example.com/commande/17'
- texte avec mention : '@client Livraison rapide et produit conforme à mes attentes.'
- texte avec hashtag : 'Très satisfait de mon achat 👍 #avis'
- texte avec emojis : 'Très satisfait de mon achat 👍 #avis'
- texte avec beaucoup de ponctuation : 'Produit excellent, je suis très satisfait. !!!'
- texte en majuscules : "LA BATTERIE TIENT VRAIMENT BIEN ET L'ÉCRAN EST SUPERBE."
- texte avec repetition de caracteres : 'Produit excellent, je suis très satisfait. !!!'


C:\Users\NGOM PC\AppData\Local\Temp\ipykernel_21060\2146999153.py:19: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  'texte avec repetition de caracteres': texte_series[texte_series.str.contains(repetition_re)].iloc[0],
